<a href="https://colab.research.google.com/github/Smeerz99/northstar-analytics-coursework/blob/main/notebooks/02_sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SQL Analysis in R

This notebook uses SQL and R to query the structured Northstar datasets. The aim is to identify patterns in delivery perfromance, operational inefficiency, and customer service outcomes by combining data across business functions.

In [12]:
!apt-get -qq update
!apt-get -qq install -y r-base r-cran-dbi r-cran-rsqlite r-cran-readr r-cran-dplyr
!pip -q install rpy2
%load_ext rpy2.ipython

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [13]:
%%R
library(DBI)
library(RSQLite)
library(readr)
library(dplyr)

## Loading datasets into R

The datasets were loaded into R and stored in a SQLite database so that SQL queries could be used to join and analyse operational records.

In [14]:
%%R
orders <- read_csv("/content/orders.csv", show_col_types = FALSE)
deliveries <- read_csv("/content/deliveries.csv", show_col_types = FALSE)
customers <- read_csv("/content/customers.csv", show_col_types = FALSE)
drivers <- read_csv("/content/drivers.csv", show_col_types = FALSE)
vehicles <- read_csv("/content/vehicles.csv", show_col_types = FALSE)
hubs <- read_csv("/content/hubs.csv", show_col_types = FALSE)
incidents <- read_csv("/content/incidents.csv", show_col_types = FALSE)
complaints <- read_csv("/content/complaints.csv", show_col_types = FALSE)

con <- dbConnect(SQLite(), ":memory:")

dbWriteTable(con, "orders", orders, overwrite = TRUE)
dbWriteTable(con, "deliveries", deliveries, overwrite = TRUE)
dbWriteTable(con, "customers", customers, overwrite = TRUE)
dbWriteTable(con, "drivers", drivers, overwrite = TRUE)
dbWriteTable(con, "vehicles", vehicles, overwrite = TRUE)
dbWriteTable(con, "hubs", hubs, overwrite = TRUE)
dbWriteTable(con, "incidents", incidents, overwrite = TRUE)
dbWriteTable(con, "complaints", complaints, overwrite = TRUE)

In [15]:
%%R
dbListTables(con)

[1] "complaints" "customers"  "deliveries" "drivers"    "hubs"      
[6] "incidents"  "orders"     "vehicles"  


## Query 1: Delivery outcomes by hubs

This query examines delivery outcomes by hub to identify whether poor performance is concentrated in specific operational locations.

In [16]:
%%R
query1 <- "
SELECT
    h.hub_name,
    h.zone,
    d.delivery_status,
    COUNT(*) AS delivery_count
FROM deliveries d
JOIN hubs h
    ON d.hub_id = h.hub_id
GROUP BY h.hub_name, h.zone, d.delivery_status
ORDER BY h.hub_name, delivery_count DESC;
"

q1_result <- dbGetQuery(con, query1)
q1_result

         hub_name      zone delivery_status delivery_count
1     Airport Hub   Airport          OnTime             62
2     Airport Hub   Airport         Delayed             27
3     Airport Hub   Airport          Failed             15
4    Central Core   Central          OnTime             67
5    Central Core   Central         Delayed             25
6    Central Core   Central          Failed             23
7       East Dock      East          OnTime             85
8       East Dock      East         Delayed             23
9       East Dock      East          Failed             11
10  Midtown Relay   Central          OnTime             80
11  Midtown Relay   Central          Failed             26
12  Midtown Relay   Central         Delayed             22
13 North Exchange     North          OnTime             93
14 North Exchange     North         Delayed             26
15 North Exchange     North          Failed             17
16  Riverside Hub Riverside          OnTime             

## Interpretation

These results suggest that delivery performance is not evenly distributed across all the hubs. Central Core and Midtown Relay seem to have weak delivery outcomes compared to other hubs with higher delayed and failed counts. In contrast, hubs such as East Dock and South Link show stronger on-time performance. This suggests that some inefficiencies may be location specific rather than being spread evenly across the whole network.

## Interpretation

This query shows how delivery outcomes are distributed across the hubs. If some hubs have a higher number of delayed or failed deliveries, it can suggest local operational issues rather than a network-wide problem.

In [17]:
%%R
query2 <- "
SELECT
    v.maintenance_status,
    d.delivery_status,
    COUNT(*) AS delivery_count
FROM deliveries d
JOIN vehicles v
    ON d.vehicle_id = v.vehicle_id
GROUP BY v.maintenance_status, d.delivery_status
ORDER BY v.maintenance_status, delivery_count DESC;
"

q2_result <- dbGetQuery(con, query2)
q2_result

  maintenance_status delivery_status delivery_count
1             Active          OnTime            384
2             Active         Delayed            113
3             Active          Failed             45
4           InRepair          OnTime            125
5           InRepair          Failed             77
6           InRepair         Delayed             52
7          Scheduled          OnTime            107
8          Scheduled         Delayed             37
9          Scheduled          Failed             10


## Interpretation

The results from query 2 show clear relationships between vehicle maintenance status and delivery performance. Deliveries completed by vehicles marked as "InRepair" have a much lower on-time count and a much higher failed count than the ones using "Active" or "Scheduled" vehicles. This suggests that vehicle condition is likely to be one of the operational factiors causing service failure and delay.

## Query 3: Complaints by delivery status

To connect operational performance with customer experience, this query compares complain volumes across different delivery outcomes.

In [18]:
%%R
query3 <- "
SELECT
    d.delivery_status,
    COUNT(c.complaint_id) AS complaint_count
FROM deliveries d
JOIN orders o
    ON d.order_id = o.order_id
LEFT JOIN complaints c
    ON o.order_id = c.order_id
GROUP BY d.delivery_status
ORDER BY complaint_count DESC;
"

q3_result <- dbGetQuery(con, query3)
q3_result

  delivery_status complaint_count
1          OnTime             149
2         Delayed              48
3          Failed              35


## Interpretation

This query shows whether customer complaints are more common when deliveries are delayed or failed. If complain counts are higher for these outcomes, it could suggest that operational performance problems are directly affecting customer satisfaction.

## Query 4: Incidents by delivery status

Incident data can help explain the operational causes behind the delays and failed deliveries. This query links incident records to delivery outcomes.

In [19]:
%%R
query4 <- "
SELECT
    d.delivery_status,
    COUNT(i.incident_id) AS incident_count
FROM deliveries d
LEFT JOIN incidents i
    ON d.delivery_id = i.delivery_id
GROUP BY d.delivery_status
ORDER BY incident_count DESC;
"

q4_result <- dbGetQuery(con, query4)
q4_result

  delivery_status incident_count
1          OnTime            191
2         Delayed             55
3          Failed             34


## Interpretation

This query helps show whether the incidents are linked with delayed or failed deliveries. Higher incident counts for weaker delivery outcomes would suggest that disruption events are a cause of poor service performance.

## Query 5: Service types linked to poor outcomes

This query combines orders and deliveries to examine whether some service types appear more likely to experience delays or failures than others.

In [20]:
%%R
query5 <- "
SELECT
    o.service_type,
    d.delivery_status,
    COUNT(*) AS delivery_count
FROM orders o
JOIN deliveries d
    ON o.order_id = d.order_id
GROUP BY o.service_type, d.delivery_status
ORDER BY o.service_type, delivery_count DESC;
"

q5_result <- dbGetQuery(con, query5)
q5_result

   service_type delivery_status delivery_count
1      Business          OnTime             73
2      Business         Delayed             28
3      Business          Failed             25
4       Medical          OnTime             70
5       Medical         Delayed             22
6       Medical          Failed             16
7        Parcel          OnTime            156
8        Parcel         Delayed             49
9        Parcel          Failed             25
10    Passenger          OnTime            171
11    Passenger         Delayed             53
12    Passenger          Failed             38
13       Retail          OnTime            146
14       Retail         Delayed             50
15       Retail          Failed             28


## Interpretation

Query 5 shows whether certain service types are associated with weaker delivery performance. If some service categories have higher delayed or failed delivery counts, it can show that those services are harder to manage operationally or require better planning and resource allocation.

## Summary of the SQL findings

The SQL analysis shows that structures operational data can be combined to identify clear performance patterns across NorthStar's network. The results suggest that poor delivery outcomes are not evenly distributed and may be linked to specific hubs, vehicle maintenance issues, or service type. These finidngs provide a structured evidence base for t he later R analytics and MongoDB sections.